# Bayes y redes bayesianas: razonar con probabilidades

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

**Facsímil 12 · Razonamiento e incertidumbre** — capítulo 2 (Bayes y redes bayesianas).

Casi nada de lo que importa es seguro: un análisis médico, un correo que quizá sea spam, una alarma
que tal vez sea un robo. El **teorema de Bayes** es la regla que dice cómo actualizar lo que creemos
cuando llega una prueba nueva, y las **redes bayesianas** lo escalan a muchas variables que se
influyen entre sí. En este cuaderno lo construyes todo a mano, con números reales, sin librerías
mágicas: verás por qué un test «99 % fiable» puede dejarte tranquilo y por qué dos testigos que
llaman cambian tanto la historia.

### Qué vas a aprender
- El **teorema de Bayes** y por qué la **tasa base** (lo raro que es algo de partida) lo decide casi todo.
- A construir un **clasificador Naive Bayes** de texto contando palabras, y a comprobar que coincide con scikit-learn.
- Qué es una **red bayesiana** y cómo hacer **inferencia exacta por enumeración** en Python puro.
- El fenómeno de **«explicar de más»** (*explaining away*): cómo una causa apaga la sospecha de otra.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico.
Todas las cifras de los textos salen de ejecutar el código; si cambias un dato, cambiarán.

### Cuánto cuesta
Unos 20 minutos. CPU; sin GPU ni claves. Solo numpy, matplotlib y scikit-learn (ya vienen en Colab).

In [ ]:
# Colab ya los trae. En tu maquina:  pip install numpy matplotlib scikit-learn
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("Listo.")

## 1. El teorema de Bayes en una línea

Bayes relaciona lo que creías **antes** de ver la prueba con lo que debes creer **después**:

$$P(H \mid E) = \frac{P(E \mid H)\,P(H)}{P(E)}$$

- $P(H)$ es la **probabilidad previa**: cuánto creías en la hipótesis $H$ antes de nada.
- $P(E \mid H)$ es la **verosimilitud**: cómo de esperable es la prueba $E$ si $H$ fuese cierta.
- $P(H \mid E)$ es la **probabilidad posterior**: tu creencia ya actualizada con la prueba.

El denominador $P(E)$ es solo una constante que normaliza para que las dos hipótesis (cierta / falsa)
sumen 1. Lo escribimos como una función y lo usamos sin misterio.

In [ ]:
def bayes(p_h, p_e_dado_h, p_e_dado_no_h):
    'Devuelve P(H|E) a partir del previa y las dos verosimilitudes.'
    p_no_h = 1 - p_h
    p_e = p_e_dado_h * p_h + p_e_dado_no_h * p_no_h   # regla de probabilidad total
    return (p_e_dado_h * p_h) / p_e

# Calentamiento: previa 50%, una prueba que sale 80% de las veces si H y 30% si no H.
print("posterior =", round(bayes(0.5, 0.8, 0.3), 4))

## 2. El test médico que engaña a la intuición

Una enfermedad afecta al **1 % de la población** (la *tasa base* o prevalencia). Hay un test con:

- **sensibilidad 99 %**: si estás enfermo, da positivo el 99 % de las veces, $P(+ \mid \text{enfermo}) = 0.99$.
- **especificidad 95 %**: si estás sano, da negativo el 95 %, así que un sano da **falso positivo** el 5 %.

Te haces el test y sale **positivo**. La intuición grita «99 %, casi seguro que estoy enfermo».
Pongamos el número real con Bayes.

In [ ]:
prevalencia  = 0.01     # P(enfermo) previa
sensibilidad = 0.99     # P(+ | enfermo)
especificidad= 0.95     # P(- | sano)  ->  P(+ | sano) = 1 - especificidad
falso_positivo = 1 - especificidad

post = bayes(prevalencia, sensibilidad, falso_positivo)
print(f"P(enfermo | test positivo) = {post:.4f}  ->  {post*100:.1f}%")
print(f"Es decir, aproximadamente 1 de cada {1/post:.1f} positivos esta realmente enfermo.")

Sorpresa: con el test positivo la probabilidad de estar enfermo es de **alrededor del 17 %**, no del 99 %.
Aproximadamente **uno de cada seis** positivos lo está de verdad. ¿Por qué? Porque la enfermedad es muy
rara: hay tantísima gente sana que, aun con solo un 5 % de falsos positivos, los sanos mal etiquetados
**superan en número** a los pocos enfermos reales. La intuición ignora la tasa base; Bayes no.

## 3. ¿Y si la enfermedad fuese más común? Barrer la prevalencia

La posterior no depende solo del test, sino de **cómo de común** es la enfermedad. Con el mismo test,
recorremos prevalencias desde 1 entre 1000 hasta 1 de cada 2 y miramos qué pasa con la creencia posterior.

In [ ]:
print("prevalencia | P(enfermo | positivo)")
for p in [0.001, 0.01, 0.05, 0.10, 0.50]:
    po = bayes(p, sensibilidad, falso_positivo)
    print(f"   {p:>6.3f}   |        {po:.4f}")

In [ ]:
prevs = np.linspace(0.001, 0.5, 300)
posts = [bayes(p, sensibilidad, falso_positivo) for p in prevs]

plt.figure(figsize=(6.2, 3.6))
plt.plot(prevs, posts, color="black")
plt.scatter([0.01], [bayes(0.01, sensibilidad, falso_positivo)], color="red", zorder=3,
            label="nuestro caso (prevalencia 1%)")
plt.xlabel("prevalencia (lo comun que es la enfermedad)")
plt.ylabel("P(enfermo | test positivo)")
plt.title("La misma prueba, distinta tasa base")
plt.ylim(0, 1.02); plt.legend(); plt.tight_layout(); plt.show()
print("Con la prueba fija, la posterior sube con la prevalencia: la tasa base manda.")

## 4. Naive Bayes: clasificar texto contando palabras

Bayes también clasifica. Para decidir si un correo es **spam** o **legítimo**, Naive Bayes calcula
$P(\text{spam} \mid \text{palabras})$ con la regla de Bayes y una suposición «ingenua» (de ahí el nombre):
que cada palabra aparece de forma **independiente** dadas la clase. Es falso —las palabras se
relacionan— pero funciona sorprendentemente bien y es rapidísimo.

Empezamos con un corpus pequeño de correos en español, etiquetados a mano.

In [ ]:
spam_train = [
    "gana dinero rapido desde casa sin esfuerzo haz clic ya",
    "has ganado un premio de mil euros reclama tu regalo ahora",
    "oferta exclusiva solo hoy descuento del noventa por ciento compra ya",
    "tu cuenta ha sido bloqueada verifica tus datos en este enlace urgente",
    "credito inmediato sin aval dinero facil en cinco minutos",
    "felicidades has sido seleccionado para un movil gratis haz clic",
    "invierte en cripto y multiplica tu dinero garantizado no lo pierdas",
    "ultima oportunidad el premio caduca hoy reclama el dinero ahora",
    "trabaja desde casa y gana mil euros a la semana sin experiencia",
    "descuento milagro solo por hoy compra ya y ahorra dinero",
]
ham_train = [
    "hola maria te paso el informe de ventas que me pediste ayer",
    "quedamos manana a las diez para revisar el proyecto",
    "adjunto la factura del mes avisame si hay alguna duda",
    "gracias por la reunion de hoy ha sido muy productiva",
    "el envio de tu pedido llegara el martes segun el seguimiento",
    "recuerda traer el portatil a la presentacion del jueves",
    "te reenvio el correo del cliente con los nuevos requisitos",
    "confirmo la cita con el dentista para el lunes por la tarde",
    "puedes revisar el documento antes de que lo enviemos",
    "feliz cumpleanos espero que pases un dia estupendo",
]
spam_test = [
    "gana dinero facil desde casa haz clic en el enlace ahora",
    "has ganado un premio reclama tu regalo gratis hoy",
    "oferta milagro solo hoy descuento enorme compra ya",
    "tu cuenta sera bloqueada verifica tus datos urgente",
]
ham_test = [
    "hola te adjunto el informe que pediste para la reunion",
    "quedamos el martes para revisar las facturas",
    "gracias por tu ayuda con el proyecto de ayer",
    "recuerda traer el portatil manana a la oficina",
]

train_txt = spam_train + ham_train
y_train   = np.array([1]*len(spam_train) + [0]*len(ham_train))   # 1=spam, 0=legitimo
test_txt  = spam_test + ham_test
y_test    = np.array([1]*len(spam_test) + [0]*len(ham_test))
print(f"Entrenamiento: {len(train_txt)} correos ({y_train.sum()} spam, {(y_train==0).sum()} legitimos)")
print(f"Prueba:        {len(test_txt)} correos")

### Convertir texto en conteos

Construimos el **vocabulario** con las palabras del entrenamiento y representamos cada correo como un
vector de **conteos** (cuántas veces sale cada palabra). Lo hacemos a mano para que no haya caja negra.

In [ ]:
def tokeniza(t):
    return t.lower().split()

vocab = sorted({w for t in train_txt for w in tokeniza(t)})
idx = {w: i for i, w in enumerate(vocab)}

def vectoriza(textos):
    X = np.zeros((len(textos), len(vocab)), dtype=int)
    for r, t in enumerate(textos):
        for w in tokeniza(t):
            if w in idx:            # palabras nuevas en test: se ignoran
                X[r, idx[w]] += 1
    return X

X_train = vectoriza(train_txt)
X_test  = vectoriza(test_txt)
print(f"Vocabulario: {len(vocab)} palabras distintas")
print(f"Matriz de entrenamiento: {X_train.shape} (correos x palabras)")

### Entrenar Naive Bayes a mano (con suavizado de Laplace)

Para cada clase $c$ guardamos:

- el **log de la previa** $\log P(c)$ = log de la fracción de correos de esa clase;
- el **log de la verosimilitud** de cada palabra $w$, con **suavizado de Laplace** ($\alpha = 1$) para que
  ninguna palabra tenga probabilidad cero:

$$P(w \mid c) = \frac{\text{cuentas}(w, c) + \alpha}{\text{total de palabras en } c + \alpha\,|V|}$$

Clasificar es sumar logaritmos (más estable que multiplicar muchos números diminutos) y quedarse con la
clase de mayor puntuación.

In [ ]:
alpha = 1.0
V = len(vocab)
clases = [0, 1]

log_prior = {}
log_veros = {}   # log P(w|c) por clase
for c in clases:
    Xc = X_train[y_train == c]
    log_prior[c] = np.log(Xc.shape[0] / X_train.shape[0])
    cuentas = Xc.sum(axis=0)                       # cuentas de cada palabra en la clase
    total_c = cuentas.sum()
    log_veros[c] = np.log((cuentas + alpha) / (total_c + alpha * V))

def predice_mano(X):
    pred = []
    for fila in X:
        puntuaciones = {c: log_prior[c] + (fila * log_veros[c]).sum() for c in clases}
        pred.append(max(puntuaciones, key=puntuaciones.get))
    return np.array(pred)

pred_train = predice_mano(X_train)
pred_test  = predice_mano(X_test)
acc_train = (pred_train == y_train).mean()
acc_test  = (pred_test == y_test).mean()
print(f"Naive Bayes a mano -> acierto en entrenamiento: {acc_train:.3f}")
print(f"Naive Bayes a mano -> acierto en prueba:        {acc_test:.3f}")
print("Predicciones de prueba (1=spam, 0=legitimo):", pred_test.tolist())
print("Etiquetas reales:                            ", y_test.tolist())

## 5. ¿Coincide con scikit-learn?

Si nuestra cuenta es correcta, debe dar **exactamente lo mismo** que `MultinomialNB` de scikit-learn
alimentado con la misma matriz de conteos y el mismo suavizado. Comparamos los parámetros aprendidos
(log de la previa y log verosimilitud) y las predicciones.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB(alpha=1.0)
nb.fit(X_train, y_train)

# sklearn ordena por clase: clases_ = [0, 1]
prior_ok = np.allclose(nb.class_log_prior_, [log_prior[0], log_prior[1]])
veros_ok = np.allclose(nb.feature_log_prob_, np.vstack([log_veros[0], log_veros[1]]))
pred_sklearn = nb.predict(X_test)
pred_ok = np.array_equal(pred_sklearn, pred_test)

print("log de la previa coinciden:     ", prior_ok)
print("log verosimilitud coinciden:", veros_ok)
print("predicciones coinciden:     ", pred_ok)
print("acierto sklearn en prueba:  ", (pred_sklearn == y_test).mean())

### Matriz de confusión

Coincidan o no con la intuición, conviene ver **cómo** se reparten los aciertos y errores. La casilla
peligrosa de un filtro de spam es **spam tratado como legítimo** (un fraude que llega a la bandeja de
entrada) o, al revés, **un correo legítimo en la carpeta de spam** (algo importante que se pierde).

In [ ]:
def matriz_confusion(real, pred):
    m = np.zeros((2, 2), dtype=int)
    for r, p in zip(real, pred):
        m[r, p] += 1
    return m

cm = matriz_confusion(y_test, pred_test)
print("                 pred:legitimo  pred:spam")
print(f"  real LEGITIMO        {cm[0,0]:>3}          {cm[0,1]:>3}")
print(f"  real SPAM            {cm[1,0]:>3}          {cm[1,1]:>3}")

fig, ax = plt.subplots(figsize=(3.6, 3.2))
ax.imshow(cm, cmap="Greys")
ax.set_xticks([0, 1]); ax.set_xticklabels(["legitimo", "spam"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["legitimo", "spam"])
ax.set_xlabel("predicho"); ax.set_ylabel("real")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
ax.set_title("Matriz de confusion"); plt.tight_layout(); plt.show()

## 6. Redes bayesianas: la red de la alarma

Bayes con una hipótesis está bien, pero la realidad tiene **muchas variables que se influyen**. Una
**red bayesiana** es un grafo donde cada nodo es una variable y cada flecha una influencia causal
directa; cada nodo trae una **tabla de probabilidad condicional** (CPT) que dice cómo depende de sus
padres. El ejemplo clásico (Russell y Norvig) es la **alarma antirrobo**:

- Puede haber un **robo** o un **terremoto** (ambos raros e independientes).
- Cualquiera de los dos puede disparar la **alarma**.
- Si suena la alarma, dos vecinos —**Juan** y **María**— quizá **llamen** por teléfono.

```
   Robo      Terremoto
      \\       /
       \\     /
        Alarma
        /    \\
       /      \\
  JuanLlama  MariaLlama
```

Cada variable es de sí/no. Definimos las CPT con los números del libro.

In [ ]:
# Probabilidades de cada nodo. 1 = si, 0 = no.
P_robo = 0.001          # P(Robo = si)
P_terremoto = 0.002     # P(Terremoto = si)

# P(Alarma = si | Robo, Terremoto)
P_alarma = {(1, 1): 0.95, (1, 0): 0.94, (0, 1): 0.29, (0, 0): 0.001}

# P(Llama = si | Alarma)
P_juan  = {1: 0.90, 0: 0.05}
P_maria = {1: 0.70, 0: 0.01}

def p_var(prob_si, valor):
    'Devuelve P(variable = valor) dado P(variable = si).'
    return prob_si if valor else 1 - prob_si

def conjunta(robo, terr, alarma, juan, maria):
    'Probabilidad conjunta de una asignacion completa de las 5 variables.'
    return (p_var(P_robo, robo)
            * p_var(P_terremoto, terr)
            * p_var(P_alarma[(robo, terr)], alarma)
            * p_var(P_juan[alarma], juan)
            * p_var(P_maria[alarma], maria))

print("Ejemplo: P(robo=no, terremoto=no, alarma=no, juan=no, maria=no) =",
      round(conjunta(0, 0, 0, 0, 0), 6))

### Inferencia exacta por enumeración

Para responder a cualquier pregunta del tipo «¿probabilidad de X dado que observo Y?» basta con
**sumar la conjunta** sobre todas las variables que no son ni la consulta ni la evidencia. Es fuerza
bruta sobre las $2^5 = 32$ combinaciones posibles, pero exacta. Primero comprobamos que la conjunta
está bien construida: **debe sumar 1** sobre todas las asignaciones.

In [ ]:
from itertools import product

VARS = ["robo", "terr", "alarma", "juan", "maria"]

def todas_las_asignaciones():
    for valores in product([0, 1], repeat=5):
        yield dict(zip(VARS, valores))

total = sum(conjunta(**a) for a in todas_las_asignaciones())
print("La conjunta suma:", round(total, 10), "(debe ser 1.0)")

In [ ]:
def consulta(objetivo, evidencia):
    'P(objetivo=si | evidencia) por enumeracion. evidencia es un dict (var: valor).'
    masa = {0: 0.0, 1: 0.0}
    for a in todas_las_asignaciones():
        if all(a[k] == v for k, v in evidencia.items()):
            masa[a[objetivo]] += conjunta(**a)
    return masa[1] / (masa[0] + masa[1])

p = consulta("robo", {"juan": 1, "maria": 1})
print(f"P(robo | Juan llama, Maria llama) = {p:.4f}  ->  {p*100:.1f}%")
print(f"Recuerda: P(robo) previa era {P_robo} ({P_robo*100:.1f}%).")

Que **los dos vecinos llamen** lleva la probabilidad de robo desde un 0,1 % previa hasta cerca del
**28 %**: multiplicada por casi 300. Pero fíjate en que sigue **lejos del 100 %**: la mayoría de las
veces que ambos llaman no hay robo, porque el robo es rarísimo de partida (otra vez la tasa base). Una
herramienta de producción como `pgmpy` hace exactamente esta cuenta de forma optimizada; aquí la ves
por dentro.

## 7. Más consultas a la misma red

Con la función de enumeración ya podemos preguntar cualquier cosa. Comparemos cómo va creciendo la
sospecha de robo según lo que observamos.

In [ ]:
casos = [
    ("sin observar nada",            {}),
    ("solo Juan llama",              {"juan": 1}),
    ("Juan y Maria llaman",          {"juan": 1, "maria": 1}),
    ("suena la alarma",              {"alarma": 1}),
    ("alarma y ademas terremoto",    {"alarma": 1, "terr": 1}),
]
print(f"{'evidencia':<28}{'P(robo | evidencia)':>20}")
for nombre, ev in casos:
    print(f"{nombre:<28}{consulta('robo', ev):>20.4f}")

## 8. Explicar de más: cuando una causa apaga la sospecha de otra

Mira las dos últimas filas de la tabla anterior. Si **suena la alarma**, la probabilidad de robo sube
mucho. Pero si **además sabemos que ha habido un terremoto**, la sospecha de robo **se desploma**: el
terremoto ya explica por sí solo la alarma, así que deja de hacer falta el robo para justificar lo que
oímos. Es el fenómeno de *explaining away* (explicar de más): dos causas que compiten por explicar un
mismo efecto.

Robo y terremoto son **independientes previa** (saber de uno no te dice nada del otro). Pero, **una vez
observada la alarma**, dejan de serlo: pasan a estar acoplados. Lo vemos con números.

In [ ]:
p_solo_alarma = consulta("robo", {"alarma": 1})
p_alarma_terr = consulta("robo", {"alarma": 1, "terr": 1})
print(f"P(robo | alarma)              = {p_solo_alarma:.4f}  ({p_solo_alarma*100:.1f}%)")
print(f"P(robo | alarma y terremoto)  = {p_alarma_terr:.4f}  ({p_alarma_terr*100:.2f}%)")
print(f"\nEl terremoto 'explica' la alarma y reduce la sospecha de robo a 1/{p_solo_alarma/p_alarma_terr:.0f}.")

# Comprobacion de independencia previa (sin observar la alarma):
p_terr = P_terremoto
p_terr_dado_robo = consulta("terr", {"robo": 1})
print(f"\nA priori son independientes: P(terremoto)={p_terr:.4f}, "
      f"P(terremoto | robo)={p_terr_dado_robo:.4f} (iguales).")

## 9. La red, dibujada

Un grafo deja clara la estructura: las causas arriba, los efectos abajo, las flechas marcando la
dirección de la influencia. Lo dibujamos a mano con matplotlib.

In [ ]:
nodos = {
    "Robo":      (0.25, 0.85),
    "Terremoto": (0.75, 0.85),
    "Alarma":    (0.50, 0.50),
    "JuanLlama": (0.25, 0.15),
    "MariaLlama":(0.75, 0.15),
}
aristas = [("Robo", "Alarma"), ("Terremoto", "Alarma"),
           ("Alarma", "JuanLlama"), ("Alarma", "MariaLlama")]

fig, ax = plt.subplots(figsize=(5.6, 4.4))
for a, b in aristas:
    xa, ya = nodos[a]; xb, yb = nodos[b]
    ax.annotate("", xy=(xb, yb), xytext=(xa, ya),
                arrowprops=dict(arrowstyle="-|>", color="gray", lw=1.6,
                                shrinkA=20, shrinkB=20))
for nombre, (x, y) in nodos.items():
    ax.scatter([x], [y], s=2600, color="white", edgecolor="black", zorder=3)
    ax.text(x, y, nombre, ha="center", va="center", fontsize=9, zorder=4)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Red bayesiana de la alarma")
plt.tight_layout(); plt.show()

## 10. Pruébalo tú

1. **Test médico menos fiable:** baja la `especificidad` a 0.90 (más falsos positivos) y recalcula la
   posterior. ¿Sube o baja la confianza tras un positivo?
2. **Solo un vecino:** calcula `consulta("robo", {"juan": 1, "maria": 0})`. ¿Cuánto pesa que María
   **no** llame?
3. **Tu propio correo:** añade una frase a `test_txt` y mira qué decide `predice_mano`.
4. **¿Y si Juan es un exagerado?** Sube `P_juan[0]` (llama aunque no haya alarma) y observa cómo
   cambia `P(robo | Juan llama)`. ¿Por qué pierde valor su llamada?

## 11. Errores comunes

- **Ignorar la tasa base.** El error estrella: confundir $P(+ \mid \text{enfermo})$ con
  $P(\text{enfermo} \mid +)$. Sin la prevalencia, el número no significa nada.
- **Olvidar el suavizado.** Sin Laplace, una palabra que nunca apareció en una clase le asigna
  probabilidad cero y **anula** todo el producto. El $\alpha$ lo evita.
- **Creer que «mucha evidencia» implica certeza.** Que dos vecinos llamen multiplica la sospecha por
  cientos, pero partiendo de 0,1 % sigue sin llegar a la mitad.
- **Confundir independencia previa con independencia siempre.** Robo y terremoto son independientes
  hasta que observas la alarma; entonces se acoplan (*explaining away*).

## 12. Qué te llevas

- El **teorema de Bayes** actualiza creencias con pruebas, y la **tasa base** (lo raro que algo es de
  partida) suele pesar más que la propia prueba.
- **Naive Bayes** clasifica texto contando palabras con suavizado de Laplace; bien hecho, **coincide al
  decimal** con scikit-learn, porque no hay magia, solo cuentas.
- Una **red bayesiana** organiza muchas variables con sus CPT, y la **inferencia por enumeración** responde
  cualquier consulta sumando la conjunta. Para redes grandes se usan algoritmos más finos (eliminación de
  variables, muestreo) y librerías como **pgmpy**, pero la idea es esta.
- *Explaining away* muestra lo sutil que es el razonamiento causal: observar una causa **cambia** lo que
  creemos de otra, aunque fueran independientes de entrada.

**Para seguir:** el capítulo siguiente del facsímil entra en modelos gráficos más generales y en cómo se
aprenden las CPT a partir de datos; y el facsímil 7 conecta esto con calibración y evaluación de
probabilidades.

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*